In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from pathlib import Path
import sys

pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

print(f"Python: {sys.executable}")

Python: /home/mirisan/github/TTC-Delay-Prediction/.venv/bin/python


In [2]:
# Toronto Open Data CKAN API — no auth needed
base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
package_id = "996cfe8d-fb35-40ce-b569-698d51fc683b"  # TTC Subway Delay Data

response = requests.get(
    f"{base_url}/api/3/action/package_show",
    params={"id": package_id}
)
package = response.json()["result"]

print(f"Dataset: {package['title']}")
print(f"\nResources available:")
for r in package["resources"]:
    print(f"  - {r['name']} ({r['format']}) — datastore_active: {r.get('datastore_active', False)}")

Dataset: TTC Subway Delay Data

Resources available:
  - ttc-subway-delay-codes (XLSX) — datastore_active: False
  - ttc-subway-delay-data-readme (XLSX) — datastore_active: False
  - ttc-subway-delay-jan-2014-april-2017 (XLSX) — datastore_active: False
  - ttc-subway-delay-may-december-2017 (XLSX) — datastore_active: False
  - ttc-subway-delay-data-2018 (XLSX) — datastore_active: False
  - ttc-subway-delay-data-2019 (XLSX) — datastore_active: False
  - ttc-subway-delay-data-2020 (XLSX) — datastore_active: False
  - ttc-subway-delay-data-2021 (XLSX) — datastore_active: False
  - ttc-subway-delay-data-2022 (XLSX) — datastore_active: False
  - ttc-subway-delay-data-2023 (XLSX) — datastore_active: False
  - ttc-subway-delay-data-2024 (XLSX) — datastore_active: False
  - TTC Subway Delay Data since 2025 (XLSX) — datastore_active: True
  - TTC Subway Delay Data since 2025.csv (CSV) — datastore_active: False
  - TTC Subway Delay Data since 2025.xml (XML) — datastore_active: False
  - TTC Subw

In [3]:
# Download the files we need


# The years we want: 2022, 2023, 2024, and 2025-so-far
# Plus the code descriptions reference

# 1. Make a set (not a list — we're checking membership, sets are faster) 
wanted_resources = {
    "ttc-subway-delay-data-2022",
    "ttc-subway-delay-data-2023",
    "ttc-subway-delay-data-2024",
    "TTC Subway Delay Data since 2025.csv",  # grab CSV version for 2025
    "Code Descriptions.csv",                 # CSV version of delay codes
}

# 2. Make sure data/raw exists (mkdir with parents=True, exist_ok=True).
dest = Path("../data/raw")
dest.mkdir(parents=True, exist_ok=True)

# 3. Loop through package["resources"]:
#    - Skip if the name isn't in our wanted set
#    - Build a target filename (sanitize spaces if needed)
#    - If file already exists, skip
#    - Otherwise: requests.get(resource["url"]), write bytes to file
for resource in package["resources"]:
    if resource["name"] not in wanted_resources:
        continue

    name = resource["name"]
    fmt = resource["format"].lower()
    
    # Build a clean filename
    safe_name = name.replace(" ", "_").replace("/", "-")
    if not safe_name.lower().endswith(f".{fmt}"):
        safe_name = f"{safe_name}.{fmt}"
    target = dest / safe_name
    
    if target.exists():
        print(f"Already exists: {target.name}")
        continue

    print(f"Downloading: {name}...")
    r = requests.get(resource["url"])
    r.raise_for_status()
    target.write_bytes(r.content)
    print(f"  → {target.name} ({len(r.content) / 1024:.1f} KB)")
    
# 4. Print the contents of data/raw at the end so we can see what landed.
print("\nFiles in data/raw/:")
for f in sorted(dest.iterdir()):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

Already exists: ttc-subway-delay-data-2022.xlsx
Already exists: ttc-subway-delay-data-2023.xlsx
Already exists: ttc-subway-delay-data-2024.xlsx
Already exists: TTC_Subway_Delay_Data_since_2025.csv
Already exists: Code_Descriptions.csv

Files in data/raw/:
  Code_Descriptions.csv (5.5 KB)
  TTC_Subway_Delay_Data_since_2025.csv (1889.9 KB)
  ttc-subway-delay-data-2022.xlsx (1061.6 KB)
  ttc-subway-delay-data-2023.xlsx (1208.1 KB)
  ttc-subway-delay-data-2024.xlsx (1390.2 KB)


In [4]:
# Check the code descriptions (reference file)

# 1. Load Code_Descriptions.csv from data/raw into a DataFrame called codes_df
codes_df = pd.read_csv("../data/raw/Code_Descriptions.csv")

# 2. Print its shape and columns
print(f"Shape: {codes_df.shape}")
print(f"Columns: {codes_df.columns.tolist()}")

# 3. Show the first 10 rows
codes_df.head(10)

Shape: (140, 3)
Columns: ['_id', 'CODE', 'DESCRIPTION']


,_id,CODE,DESCRIPTION
0,1,EUAC,AIR CONDITIONING
1,2,EUAL,ALTERNATING CURRENT
2,3,EUATC,ATC RC&S EQUIPMENT
3,4,EUBK,BRAKES
4,5,EUBO,BODY
5,6,EUCA,COMPRESSED AIR
6,7,EUCC,CAM CONTROL
7,8,EUCD,RC&S CONSEQUENTIAL DELAY (SECOND DELAY SAME FA...
8,9,EUCH,CHOPPER CONTROL
9,10,EUCO,COUPLERS


In [5]:
# Load one year of delay data to inspect schema

# 1. Load ttc-subway-delay-data-2024.xlsx into a DataFrame called df_2024
df_2024 = pd.read_excel("../data/raw/ttc-subway-delay-data-2024.xlsx")

# 2. Print the shape and column list
print(f"Shape: {df_2024.shape}")
print(f"Columns: {df_2024.columns.tolist()}")
print()

# 3. Call df_2024.info() to see dtypes and null counts
df_2024.info()

Shape: (26467, 10)
Columns: ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26467 entries, 0 to 26466
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       26467 non-null  datetime64[ns]
 1   Time       26467 non-null  object        
 2   Day        26467 non-null  object        
 3   Station    26467 non-null  object        
 4   Code       26467 non-null  object        
 5   Min Delay  26467 non-null  int64         
 6   Min Gap    26467 non-null  int64         
 7   Bound      16947 non-null  object        
 8   Line       26423 non-null  object        
 9   Vehicle    26467 non-null  int64         
dtypes: datetime64[ns](1), int64(3), object(6)
memory usage: 2.0+ MB


In [6]:
df_2024.head(10)

,Date,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Line,Vehicle
0,2024-01-01,02:00,Monday,SHEPPARD STATION,MUI,0,0,N,YU,5491
1,2024-01-01,02:00,Monday,DUNDAS STATION,MUIS,0,0,N,YU,0
2,2024-01-01,02:08,Monday,DUNDAS STATION,MUPAA,4,10,N,YU,6051
3,2024-01-01,02:13,Monday,KENNEDY BD STATION,PUTDN,10,16,E,BD,5284
4,2024-01-01,02:22,Monday,BLOOR STATION,MUPAA,4,10,N,YU,5986
5,2024-01-01,02:25,Monday,ST CLAIR STATION,MUPAA,3,9,N,YU,6051
6,2024-01-01,02:25,Monday,BLOOR STATION,MUIRS,0,0,S,YU,0
7,2024-01-01,02:27,Monday,WOODBINE STATION,EUDO,7,13,E,BD,5077
8,2024-01-01,02:28,Monday,FINCH STATION,MUIRS,0,0,S,YU,5561
9,2024-01-01,02:30,Monday,DAVISVILLE STATION,MUI,13,19,N,YU,6051


In [9]:
# Quick schema check across all 4 years
years_files = {
    2022: "../data/raw/ttc-subway-delay-data-2022.xlsx",
    2023: "../data/raw/ttc-subway-delay-data-2023.xlsx",
    2024: "../data/raw/ttc-subway-delay-data-2024.xlsx",
    2025: "../data/raw/TTC_Subway_Delay_Data_since_2025.csv",
}

for year, fp in years_files.items():
    d = pd.read_excel(fp) if fp.endswith(".xlsx") else pd.read_csv(fp)
    print(f"{year}: {d.shape} — {d.columns.tolist()}")

2022: (19895, 10) — ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
2023: (22949, 10) — ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
2024: (26467, 10) — ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']
2025: (28191, 11) — ['_id', 'Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle']


In [12]:
#Combine all 4 years into one DataFrame.

# 1. Start with an empty list called `dfs` (will hold each year's DataFrame).
dfs = []

# 2. Loop through years_files.items(). For each (year, filepath):
#    - Load with pd.read_excel if filepath ends in .xlsx, else pd.read_csv
#    - Drop the '_id' column if it exists:
#        d = d.drop(columns=['_id'], errors='ignore')
#      (2025 has it, others don't — errors='ignore' handles both cases)
#    - Add: d['source_year'] = year
#    - Append d to dfs
for year, fp in years_files.items():
    d = pd.read_excel(fp) if fp.endswith(".xlsx") else pd.read_csv(fp)
    d = d.drop(columns=['_id'], errors='ignore')
    d["source_year"] = year
    dfs.append(d)


# 3. Concatenate: df = pd.concat(dfs, ignore_index=True)
df = pd.concat(dfs, ignore_index=True)

# Normalize Date column — forces strings and timestamps into consistent datetime64

df['Date'] = pd.to_datetime(df['Date'])

# 4. Print:
#    - Combined shape
#    - Date range (df['Date'].min() and max())
#    - Rows per year: df['source_year'].value_counts().sort_index()
#    - Final column list
print(f"Combined shape: {df.shape}")
print(f"Date range: {df['Date'].min()} → {df['Date'].max()}")
print(f"\nRows per source year:")
print(df['source_year'].value_counts().sort_index())
print(f"\nColumns: {df.columns.tolist()}")

Combined shape: (97502, 11)
Date range: 2022-01-01 00:00:00 → 2026-01-31 00:00:00

Rows per source year:
source_year
2022    19895
2023    22949
2024    26467
2025    28191
Name: count, dtype: int64

Columns: ['Date', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Line', 'Vehicle', 'source_year']


In [14]:
df['datetime_str'] = df['Date'].dt.strftime('%Y-%m-%d') + ' ' + df['Time']
df['datetime'] = pd.to_datetime(df['datetime_str'], errors='coerce')
df = df.drop(columns=['datetime_str'])

n_bad = df['datetime'].isna().sum()
print(f"Rows with unparseable datetime: {n_bad}")
print(f"New datetime column dtype: {df['datetime'].dtype}")
print(f"\nSample:")
df[['Date', 'Time', 'datetime']].head(10)

Rows with unparseable datetime: 0
New datetime column dtype: datetime64[ns]

Sample:


,Date,Time,datetime
0,2022-01-01,15:59,2022-01-01 15:59:00
1,2022-01-01,02:23,2022-01-01 02:23:00
2,2022-01-01,22:00,2022-01-01 22:00:00
3,2022-01-01,02:28,2022-01-01 02:28:00
4,2022-01-01,02:34,2022-01-01 02:34:00
5,2022-01-01,05:40,2022-01-01 05:40:00
6,2022-01-01,06:56,2022-01-01 06:56:00
7,2022-01-01,06:58,2022-01-01 06:58:00
8,2022-01-01,07:01,2022-01-01 07:01:00
9,2022-01-01,07:43,2022-01-01 07:43:00


In [15]:
output_path = Path("../data/processed/ttc_delays_combined.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_parquet(output_path, index=False)
print(f"Saved to {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")

# Roundtrip check
test = pd.read_parquet(output_path)
print(f"\nRoundtrip loaded: shape={test.shape}")
print(f"datetime dtype preserved: {test['datetime'].dtype}")

ArrowKeyError: No type extension with name arrow.py_extension_type found